In [1]:
import os
import numpy as np
import torch
import torch.nn as nn
from data_provider.data_factory import data_provider
from types import SimpleNamespace

In [2]:
configs_dict = {
    'task_name': 'long_term_forecast',  
    'root_path': './dataset/ETT-small',
    'data_path': 'ETTm1.csv',
    'model_id': 'test',
    'model': 'BDLinear',
    'data': 'custom',
    'features': 'M',
    'seq_len': 32,
    'pred_len': 32,
    'enc_in': 7,
    'dec_in': 7,
    'c_out': 7,
    'd_model': 512,
    'embed_dim': 512,
    'revin': 0,
    'add_module': 'embed',
    'iter_norm': 1,
    'affine': 0,
    'eps': 1e-5,
    'T': 5,
    'num_groups': 1,
    'label_len': 16,
    'target': 'OT',
    'freq': 'h',
    'embed': 'timeF',
    'batch_size': 4,
    'num_workers': 10,
    'add_noise': False,
    'noise_amp': 1,
    'noise_freq_percentage': 0.05,
    'noise_seed': 2023,
    'noise_type': 'sin',
    'data_percentage': 1.0,
    'seasonal_patterns': 'Monthly',
}

In [3]:
configs = SimpleNamespace(**configs_dict)

In [4]:
train_data, train_loader = data_provider(configs, flag='train')
vali_data, vali_loader = data_provider(configs, flag='val')
test_data, test_loader = data_provider(configs, flag='test')


Head lines of raw dataframe:
                  date   HUFL   HULL   MUFL   MULL   LUFL   LULL         OT
0  2016-07-01 00:00:00  5.827  2.009  1.599  0.462  4.203  1.340  30.531000
1  2016-07-01 00:15:00  5.760  2.076  1.492  0.426  4.264  1.401  30.459999
2  2016-07-01 00:30:00  5.760  1.942  1.492  0.391  4.234  1.310  30.038000
3  2016-07-01 00:45:00  5.760  1.942  1.492  0.426  4.234  1.310  27.013000
4  2016-07-01 01:00:00  5.693  2.076  1.492  0.426  4.142  1.371  27.787001
train 48713
val 6937
test 13905


In [5]:
batch_x, batch_y, batch_x_mark, batch_y_mark = next(iter(train_loader))

print(batch_x.shape)
print(batch_y.shape)
print(batch_x_mark.shape)
print(batch_y_mark.shape)

torch.Size([4, 32, 7])
torch.Size([4, 48, 7])
torch.Size([4, 32, 4])
torch.Size([4, 48, 4])


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [7]:
batch_x = batch_x.float().to(device)
batch_y = batch_y.float().to(device)
batch_x_mark = batch_x_mark.float().to(device)
batch_y_mark = batch_y_mark.float().to(device)

In [8]:
dec_inp = torch.zeros_like(batch_y[:, -configs.pred_len:, :]).float()
dec_inp = torch.cat([batch_y[:, :configs.label_len, :], dec_inp], dim=1).float().to(device)


In [9]:
class iterative_normalization_sequential(torch.autograd.Function):

    debug_outputs = []

    @staticmethod
    def forward(ctx, *args, **kwargs):
        X, running_mean, running_wmat, nc, ctx.T, eps, momentum, training = args
        debug_info = {}
        # change NxCxL to (G x D) x(NxL), i.e., g*d*m
        # X: [N, C, L] -> x: [g, nc, m] where m = N*L
        # print('iterNorm_input_x shape:', X.shape)
        # print('iterNorm_input_x:', X)
        ctx.g = X.size(1) // nc                                                     # group数
        x = X.transpose(0, 1).contiguous().view(ctx.g, nc, -1)                      # [g, nc, m]
        # print('iterNorm_reshape_x shape:', x.shape)
        # print('iterNorm_reshape_x:', x)
        _, d, m = x.size()                                                           # d: channel数, m: 样本数
        saved = []
        if training:
            # calculate centered activation by subtracted mini-batch mean
            mean = x.mean(-1, keepdim=True)                                        # [g, nc, 1]
            # print('iterNorm_mean shape:', mean.shape)
            # print('iterNorm_mean:', mean)
            xc = x - mean                                                           # [g, nc, m]
            # print('iterNorm_xc shape:', xc.shape)
            # print('iterNorm_xc:', xc)
            saved.append(xc)                                                        # save[0]: Xc
            # calculate covariance matrix
            P = [None] * (ctx.T + 1)
            P[0] = torch.eye(d).to(X).expand(ctx.g, d, d)                           # P[0]: 单位矩阵[g, nc, nc]
            Sigma = torch.baddbmm(P[0], xc, xc.transpose(1, 2), beta=eps, alpha=1. / m)        # 计算协方差矩阵 [g, nc, nc]
            # print('iterNorm_ori_Sigma shape:', Sigma.shape)
            # print('iterNorm_ori_Sigma:', Sigma)
            # reciprocal of trace of Sigma: shape [g, 1, 1]
            rTr = (Sigma * P[0]).sum((1, 2), keepdim=True).reciprocal_()            # Sigma矩阵迹的倒数，shape为 [g, 1, 1]
            saved.append(rTr)                                                       # saved[1]: rTr(1/tr(sigma))
            Sigma_N = Sigma * rTr                                                   # SigmaN: [g, nc, nc]
            # print('Sigma_N:', Sigma_N.size())
            saved.append(Sigma_N)                                                   # saved[2]: SigmaN
            for k in range(ctx.T):
                P[k + 1] = torch.baddbmm(P[k], torch.matrix_power(P[k], 3), Sigma_N, beta=1.5, alpha=-0.5)         # Iter Calculate
            # 保存P的副本以避免内存共享问题
            saved.extend([p.clone() for p in P])                                         # save[3:]: Pk = SigmaN^{-1/2} [g, nc, nc]
            wm = P[ctx.T].clone().mul_(rTr.sqrt())                                          # wm = sigma^{-1/2} [g, nc, nc]
            # print('iterNorm_wm shape:', wm.shape)
            # print('iterNorm_wm:', wm)

            debug_info['mean'] = mean.detach().cpu().numpy()
            debug_info['xc'] = xc.detach().cpu().numpy()
            debug_info['Sigma'] = Sigma.detach().cpu().numpy()
            debug_info['rTr'] = rTr.detach().cpu().numpy()
            debug_info['Sigma_N'] = Sigma_N.detach().cpu().numpy()
            debug_info['wm'] = wm.detach().cpu().numpy()
            debug_info['P'] = [p.detach().cpu().numpy() for p in P]

            running_mean.copy_(momentum * mean + (1. - momentum) * running_mean)    # 更新running_mean
            running_wmat.copy_((momentum * wm + (1. - momentum) * running_wmat).clone())      # 更新running_wm
        else:
            xc = x - running_mean
            wm = running_wmat
        xn = wm.matmul(xc)                                                          # [g, nc, m] /hatX=Sigma^-1/2 * Xc
        debug_info['xn'] = xn.detach().cpu().numpy()
        # print('iterNorm_xn shape:', xn.shape)
        # print('iterNorm_xn:', xn)
        # 恢复为时序数据格式: [g, nc, m] -> [N, C, L]
        Xn = xn.view(X.size(1), X.size(0), X.size(2)).transpose(0, 1).contiguous()  # [N, C, L]
        debug_info['Xn'] = Xn.detach().cpu().numpy()

        iterative_normalization_sequential.debug_outputs.append(debug_info)
        
        # print('iterNorm_Xn shape:', Xn.shape)
        # print('iterNorm_Xn:', Xn)
        ctx.save_for_backward(*saved)
        return Xn

    @staticmethod
    def backward(ctx, *grad_outputs):
        grad, = grad_outputs                                                        # grad_Xn, [N, C, L]
        saved = ctx.saved_variables             
        xc = saved[0]                                                               # saved[0]: xc, [g, nc, m]
        rTr = saved[1]                                                              # saved[1]: rTr, [g, 1, 1]
        sn = saved[2].transpose(-2, -1)                                             # saved[2]: SigmaN^T, [g, nc, nc]
        P = saved[3:]  # saved[3:]: Pk = SigmaN^{-1/2} [g, nc, nc]                  # P[0] 到 P[T]，每个shape为 [g, nc, nc]    
        g, d, m = xc.size()

        g_ = grad.transpose(0, 1).contiguous().view_as(xc)                          # [g, nc, m]
        g_wm = g_.matmul(xc.transpose(-2, -1))                                      # ∂L/∂wm = g_ × xc^T, shape为 [g, nc, nc]
        g_P = g_wm * rTr.sqrt()                                                     # ∂L/∂P_T = ∂L/∂wm * sqrt(rTr), shape为 [g, nc, nc]
        wm = P[ctx.T]                                                               # P[T] = SigmaN^{-1/2}, [g, nc, nc]      
        g_sn = 0
        for k in range(ctx.T, 1, -1):
            P[k - 1].transpose_(-2, -1)
            P2 = P[k - 1].matmul(P[k - 1])
            g_sn += P2.matmul(P[k - 1]).matmul(g_P)
            g_tmp = g_P.matmul(sn)
            g_P.baddbmm_(g_tmp, P2, beta=1.5, alpha=-0.5)
            g_P.baddbmm_(P2, g_tmp, beta=1, alpha=-0.5)
            g_P.baddbmm_(P[k - 1].matmul(g_tmp), P[k - 1], beta=1, alpha=-0.5)
        g_sn += g_P
        # g_sn = g_sn * rTr.sqrt()
        g_tr = ((-sn.matmul(g_sn) + g_wm.transpose(-2, -1).matmul(wm)) * P[0]).sum((1, 2), keepdim=True) * P[0]
        g_sigma = (g_sn + g_sn.transpose(-2, -1) + 2. * g_tr) * (-0.5 / m * rTr)
        # g_sigma = g_sigma + g_sigma.transpose(-2, -1)
        g_x = torch.baddbmm(wm.matmul(g_ - g_.mean(-1, keepdim=True)), g_sigma, xc, beta=0, alpha=1)
        # 恢复为时序数据格式: [g, nc, m] -> [N, C, L]
        grad_input = g_x.view(grad.size(1), grad.size(0), grad.size(2)).transpose(0, 1).contiguous()    # 最终输入梯度，恢复为原始shape [N, C, L]
        return grad_input, None, None, None, None, None, None, None


In [10]:
class IterNormSequential(torch.nn.Module):
    def __init__(self, num_features, num_groups=1, num_channels=None, T=5, dim=3, eps=1e-5, momentum=0.1, affine=True,
                 *args, **kwargs):
        super(IterNormSequential, self).__init__()
        # assert dim == 3, 'IterNormSequential is designed for sequential data (NCL format)'
        self.T = T
        self.eps = eps
        self.momentum = momentum
        self.num_features = num_features        #  num_channels: 每组的通道数
        self.affine = affine
        self.dim = dim
        if num_channels is None:
            num_channels = (num_features - 1) // num_groups + 1
        num_groups = num_features // num_channels
        while num_features % num_channels != 0:
            num_channels //= 2
            num_groups = num_features // num_channels
        assert num_groups > 0 and num_features % num_groups == 0, "num features={}, num groups={}".format(num_features,
            num_groups)
        self.num_groups = num_groups
        self.num_channels = num_channels
        shape = [1] * dim
        shape[1] = self.num_features                            # shape为 [1, num_features, 1] dim=3
        if self.affine:
            self.weight = Parameter(torch.Tensor(*shape))
            self.bias = Parameter(torch.Tensor(*shape))
        else:
            self.register_parameter('weight', None)
            self.register_parameter('bias', None)

        self.register_buffer('running_mean', torch.zeros(num_groups, num_channels, 1))      # [num_groups, num_channels, 1]
        # running whiten matrix
        self.register_buffer('running_wm', torch.eye(num_channels).repeat(num_groups, 1, 1))      # [num_groups, num_channels, num_channels]
        self.reset_parameters()

    def reset_parameters(self):
        # self.reset_running_stats()
        if self.affine:
            torch.nn.init.ones_(self.weight)
            torch.nn.init.zeros_(self.bias)

    def forward(self, X: torch.Tensor):
        """
        Args:
            X: Input tensor of shape [N, C, L] where:
               N: batch size
               C: number of features/channels
               L: sequence length
        Returns:
            Normalized tensor of same shape as input
        """
        X_hat = iterative_normalization_sequential.apply(X, self.running_mean, self.running_wm, self.num_channels, self.T,
                                                 self.eps, self.momentum, self.training)
        # affine
        if self.affine:
            return X_hat * self.weight + self.bias
        else:
            return X_hat

    def extra_repr(self):
        return '{num_features}, num_channels={num_channels}, T={T}, eps={eps}, ' \
               'momentum={momentum}, affine={affine}'.format(**self.__dict__)

In [11]:
class Model(nn.Module):
    """
    Paper link: https://arxiv.org/pdf/2205.13504.pdf
    """

    def __init__(self, configs, individual=False):
        """
        individual: Bool, whether shared model among different variates.
        """
        super(Model, self).__init__()
        self.task_name = configs.task_name
        self.seq_len = configs.seq_len
        self.embed_dim = configs.d_model
        if self.task_name == 'classification' or self.task_name == 'anomaly_detection' or self.task_name == 'imputation':
            self.pred_len = configs.seq_len
        else:
            self.pred_len = configs.pred_len
        self.individual = individual
        self.channels = configs.enc_in
        self.revin = configs.revin
        self.add_module = configs.add_module

        if self.individual:
            if configs.iter_norm:
                print('iter_norm in individual mode is not supported yet')
                self.iter_norm = None
            self.Linear = nn.ModuleList()


            for i in range(self.channels):
                self.Linear.append(
                    nn.Linear(self.seq_len, self.pred_len))
                self.Linear[i].weight = nn.Parameter(
                    (1 / self.seq_len) * torch.ones([self.pred_len, self.seq_len]))

        else:
            if configs.iter_norm:
                self.iter_norm = IterNormSequential(self.seq_len, num_groups=configs.num_groups, T=configs.T, eps=configs.eps, affine=configs.affine)
            else:
                self.iter_norm = None
            
            if self.add_module == 'embed':
                self.embed = nn.Linear(self.seq_len, self.embed_dim)
                self.Linear = nn.Linear(self.embed_dim, self.pred_len)
            else:
                self.Linear = nn.Linear(self.seq_len, self.pred_len)

            # scale = 1 / self.seq_len
            # self.Linear.weight = nn.Parameter(scale * torch.ones([self.pred_len, self.embed_dim]))


        if self.task_name == 'classification':
            self.act = F.gelu
            self.dropout = nn.Dropout(configs.dropout)
            self.projection = nn.Linear(
                configs.enc_in * configs.seq_len, configs.num_class)

    def encoder(self, x):
        # x shape: [B, L, N]
        # 在N维度进行RevIN标准化
        # print('input_x shape:', x.shape)
        # print('input_x:', x)
        if self.revin:
            # x = x.permute(0, 2, 1)
            mean = x.mean(dim=1, keepdim=True)
            std = x.std(dim=1, keepdim=True)
            x = (x - mean) / (std + 1e-5)
            # x = x.permute(0, 2, 1)

        init = x.permute(0, 2, 1)   # [B, N, L]
        

        if self.individual:
            print('individual mode is not supported yet')
            output = torch.zeros(
                [init.size(0), init.size(1), self.pred_len], dtype=init.dtype
            ).to(init.device)
            for i in range(self.channels):
                output[:, i, :] = self.Linear[i](init[:, i, :])
        else:
            if self.iter_norm:
                init = self.iter_norm(init.permute(0, 2, 1)).permute(0, 2, 1)     # input shape: [B, N, D]
            if self.add_module == 'embed':
                init = self.embed(init)     # [B, N, D]
            output = self.Linear(init)

        x = output

        # 在N维度进行RevIN逆标准化
        if self.revin:
            x = x.permute(0, 2, 1)
            x = x * (std + 1e-5) + mean
            x = x.permute(0, 2, 1)

        return x.permute(0, 2, 1)

    def forecast(self, x_enc):
        # Encoder
        return self.encoder(x_enc)

    def imputation(self, x_enc):
        # Encoder
        return self.encoder(x_enc)

    def anomaly_detection(self, x_enc):
        # Encoder
        return self.encoder(x_enc)

    def classification(self, x_enc):
        # Encoder
        enc_out = self.encoder(x_enc)
        # Output
        # (batch_size, seq_length * d_model)
        output = enc_out.reshape(enc_out.shape[0], -1)
        # (batch_size, num_classes)
        output = self.projection(output)
        return output

    def forward(self, x_enc, x_mark_enc, x_dec, x_mark_dec, mask=None):
        if self.task_name == 'long_term_forecast' or self.task_name == 'short_term_forecast':
            dec_out = self.forecast(x_enc)
            return dec_out[:, -self.pred_len:, :]  # [B, L, D]
        if self.task_name == 'imputation':
            dec_out = self.imputation(x_enc)
            return dec_out  # [B, L, D]
        if self.task_name == 'anomaly_detection':
            dec_out = self.anomaly_detection(x_enc)
            return dec_out  # [B, L, D]
        if self.task_name == 'classification':
            dec_out = self.classification(x_enc)
            return dec_out  # [B, N]
        return None

In [12]:
model = Model(configs).to(device)


In [ ]:
batch_x

In [ ]:
outputs = model(batch_x, batch_x_mark, dec_inp, batch_y_mark)
debugs = iterative_normalization_sequential.debug_outputs
for i, debug in enumerate(debugs):
    print(f"Step {i}:")
    for k, v in debug.items():
        print(f"{k}: shape={np.shape(v)}")
        # 或保存到文件
        # np.save(f"{k}_step{i}.npy", v)

Step 0:
mean: shape=(1, 32, 1)
xc: shape=(1, 32, 28)
Sigma: shape=(1, 32, 32)
rTr: shape=(1, 1, 1)
Sigma_N: shape=(1, 32, 32)
wm: shape=(1, 32, 32)
P: shape=(6, 1, 32, 32)
xn: shape=(1, 32, 28)
Xn: shape=(4, 32, 7)


: 